# LLM QLoRA Fine-Tuning on SG Hate Speech Dataset (Colab)

Fine-tune two modern LLMs on the SG synthetic hate speech dataset using **Unsloth + QLoRA** on Google Colab.

**Runtime:** Colab Pro — A100 40GB (minimum) or H100 80GB (recommended)  
**Task:** 3-class classification — `hate`, `offensive`, `normal`  
**Dataset:** SG synthetic, 319 train / 106 val / 108 test  
**Stack:** Unsloth + TRL SFTTrainer + 4-bit QLoRA

## Models

| Part | Model | Size | Active params | VRAM (4-bit QLoRA) | Notes |
|------|-------|------|---------------|---------------------|-------|
| 2 | `openai/gpt-oss-safeguard-20b` | 21B MoE | 3.6B | ~14 GB | Safety reasoning model, Apache 2.0, Unsloth-supported |
| 3 | `aisingapore/Gemma-SEA-LION-v4-27B-IT` | 27B dense | 27B | ~22 GB | SEA-focused, Gemma 3 architecture, Unsloth-supported |

## Key Notes

**GPT-OSS-Safeguard** uses OpenAI's **Harmony response format** — training data must be formatted
with `<|start|>analysis<|message|>...<|end|>` and `<|start|>final<|message|>...<|end|>` channels.
This gives the model a chain-of-thought reasoning step before outputting the label.

**Gemma-SEA-LION-v4-27B-IT** is a Gemma 3 27B model post-trained on 10M SEA-language QA pairs.
Unsloth supports Gemma 3 27B directly — we attempt to load SEA-LION directly first,
with a fallback to the Unsloth Gemma 3 27B 4-bit base if the architecture check fails.

## Structure
- **Part 0:** Colab setup (run once)
- **Part 1:** Data preparation — two formats (Harmony for GPT-OSS, chat for SEA-LION)
- **Part 2:** GPT-OSS-Safeguard-20B — train → eval → clear GPU
- **Part 3:** Gemma-SEA-LION-v4-27B-IT — train → eval → clear GPU
- **Part 4:** Leaderboard

> Parts 2 and 3 are fully independent. Run either without the other.

---
# Part 0: Colab Setup

In [ ]:
# Cell 0-1: GPU Verification
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), 'No GPU — change runtime to GPU (A100 or H100)'

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()

print(f'GPU:    {gpu_name}')
print(f'VRAM:   {gpu_mem_gb:.1f} GB')
print(f'BF16:   {bf16_ok}')

if gpu_mem_gb < 35:
    print('WARNING: < 35 GB VRAM detected.')
    print('  GPT-OSS-Safeguard needs ~14 GB (OK on T4 with tight settings)')
    print('  Gemma-SEA-LION-27B needs ~22 GB (requires A100 40GB minimum)')

In [ ]:
# Cell 0-2: Mount Google Drive (separate cell)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
%%bash
# Cell 0-3: Install Unsloth + dependencies
# Unsloth auto-detects GPU (A100 uses cu121-ampere, H100 uses cu121-ampere too)
pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
pip install -q trl bitsandbytes accelerate scikit-learn pandas
echo "Install complete."

In [ ]:
# Cell 0-4: Path setup
from pathlib import Path

try:
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    assert DRIVE_ROOT.exists()
    DAP_HS_DIR  = DRIVE_ROOT / 'dap-hs'
    DATA_DIR    = DAP_HS_DIR / 'data' / 'singlish' / 'synthetic' / '20_terms' / 'split_60_20_20'
    OUTPUT_BASE = DAP_HS_DIR / 'lora_runs'
    RESULTS_FILE = str(DAP_HS_DIR / 'results' / 'model_scores_sg.csv')
    print(f'Drive root: {DAP_HS_DIR}')
except Exception:
    DAP_HS_DIR   = Path('.')
    DATA_DIR     = Path('data/singlish/synthetic/20_terms/split_60_20_20')
    OUTPUT_BASE  = Path('runs/lora')
    RESULTS_FILE = 'results/model_scores_sg.csv'
    print('Drive not available, using local paths.')

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
Path(RESULTS_FILE).parent.mkdir(parents=True, exist_ok=True)

for split in ['train', 'val', 'test']:
    p = DATA_DIR / f'{split}.jsonl'
    status = 'OK' if p.exists() else 'MISSING — copy repo data/ folder to Drive'
    print(f'  {split}.jsonl: {status}')

In [ ]:
# Cell 0-5: HuggingFace token (not required for either model — both are public)
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
    if HF_TOKEN:
        print('HF_TOKEN loaded from Colab secrets.')
    else:
        print('No HF_TOKEN found — both models are public so this is fine.')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    print('HF_TOKEN from env (empty = unauthenticated, OK for public models).')

---
# Part 1: Data Preparation

Two format functions:
- **Harmony format** — required for GPT-OSS-Safeguard (OpenAI's structured channel format)
- **Chat format** — standard for Gemma-SEA-LION

Run once — both models load from the same saved JSONL files.

In [ ]:
# Cell 1-1: Load SG splits
import json
from collections import Counter

LABEL_NAMES = ['hate', 'offensive', 'normal']

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

sg_train = load_jsonl(DATA_DIR / 'train.jsonl')
sg_val   = load_jsonl(DATA_DIR / 'val.jsonl')
sg_test  = load_jsonl(DATA_DIR / 'test.jsonl')

print(f'Train: {len(sg_train)}  Val: {len(sg_val)}  Test: {len(sg_test)}')
print(f'Label distribution (train): {dict(Counter(d["label"] for d in sg_train))}')

In [ ]:
# Cell 1-2: Shared system prompt
SYSTEM_PROMPT = """You are a Singapore content moderation expert. Classify the following text into exactly one of:

- 'hate': targets a person or group based on race, religion, nationality, or identity. Includes Singlish racial slurs (e.g. 'ah tiong', 'bangla', 'ang moh' used offensively) and xenophobic statements.
- 'offensive': rude or inappropriate language (e.g. Singlish expletives like 'nabeh', 'cb', 'wah lan', 'knn') that is NOT directed at an identity group.
- 'normal': neutral or benign content, including everyday casual Singlish ('lah', 'leh', 'shiok', 'kiasu').

Reply with ONLY the label in lowercase: hate, offensive, or normal."""

print('System prompt defined.')

In [ ]:
# Cell 1-3: Harmony format for GPT-OSS-Safeguard
#
# GPT-OSS uses OpenAI's Harmony response format. The assistant turn has two channels:
#   analysis  — chain-of-thought reasoning (internal, not shown to user)
#   final     — the actual output label
#
# Format: <|start|>{role}<|message|>{content}<|end|>
#
# We generate short synthetic reasoning for the analysis channel.
# This teaches the model to reason before classifying — critical for edge cases.

HATE_REASONING = {
    'hate':      'The text contains a generalisation or slur targeting an ethnic/national group. This is hate speech.',
    'offensive': 'The text uses Singlish expletives or insults but does not target an identity group. This is offensive, not hate.',
    'normal':    'The text is casual Singlish or benign everyday content. No hate speech or offensive language detected.',
}


def to_harmony_format(row):
    """
    Convert a SG data row to Harmony format for GPT-OSS-Safeguard SFT.
    The model learns to: (1) reason via the analysis channel, (2) output the label via final.
    """
    label = row['label']
    reasoning = HATE_REASONING[label]

    # Harmony assistant turn: analysis channel + final channel
    assistant_content = (
        f'<|start|>analysis<|message|>{reasoning}<|end|>'
        f'<|start|>final<|message|>{label}<|end|>'
    )

    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': f'Text: "{row["text"]}"'},
            {'role': 'assistant', 'content': assistant_content},
        ]
    }


# Preview
sample = to_harmony_format(sg_train[0])
print('Harmony format sample:')
for msg in sample['messages']:
    print(f'  [{msg["role"]}]: {msg["content"][:120]}')

In [ ]:
# Cell 1-4: Standard chat format for Gemma-SEA-LION

def to_chat_format(row):
    """Standard chat messages format for Gemma-SEA-LION SFT."""
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': f'Text: "{row["text"]}"'},
            {'role': 'assistant', 'content': row['label']},
        ]
    }


# Save both formats to /content for fast local access during training
FORMATTED_DIR = Path('/content/sg_formatted')
FORMATTED_DIR.mkdir(exist_ok=True)

format_configs = [
    ('harmony', to_harmony_format),
    ('chat', to_chat_format),
]

for fmt_name, fmt_fn in format_configs:
    for split_name, split_data in [('train', sg_train), ('val', sg_val)]:
        out = FORMATTED_DIR / f'{fmt_name}_{split_name}.jsonl'
        with open(out, 'w') as f:
            for row in split_data:
                f.write(json.dumps(fmt_fn(row)) + '\n')
        print(f'Saved {len(split_data)} rows → {out.name}')

In [ ]:
# Cell 1-5: Load into HuggingFace datasets
from datasets import load_dataset

harmony_dataset = load_dataset('json', data_files={
    'train': str(FORMATTED_DIR / 'harmony_train.jsonl'),
    'validation': str(FORMATTED_DIR / 'harmony_val.jsonl'),
})

chat_dataset = load_dataset('json', data_files={
    'train': str(FORMATTED_DIR / 'chat_train.jsonl'),
    'validation': str(FORMATTED_DIR / 'chat_val.jsonl'),
})

print('Harmony dataset:', harmony_dataset)
print('Chat dataset:   ', chat_dataset)

---
# Part 2: GPT-OSS-Safeguard-20B QLoRA

`openai/gpt-oss-safeguard-20b` — 21B MoE safety reasoning model (3.6B active params)  
**License:** Apache 2.0  
**VRAM:** ~14 GB for QLoRA (MoE: only active params loaded)  
**Unsloth:** Fully supported — official Colab notebook available  
**Reference:** [Unsloth gpt-oss guide](https://unsloth.ai/blog/gpt-oss)  
**Estimated time:** ~25 min on A100, ~12 min on H100

In [ ]:
# Cell 2-1: Config
MODEL_A      = 'openai/gpt-oss-safeguard-20b'
ADAPTER_A    = OUTPUT_BASE / 'gpt-oss-safeguard-sg'
ADAPTER_A.mkdir(parents=True, exist_ok=True)

# LoRA config — higher r for a 20B model on classification
LORA_R_A       = 32
LORA_ALPHA_A   = 32
LORA_DROPOUT   = 0.05
MAX_SEQ_LEN_A  = 512   # Harmony format adds tokens; keep short for small dataset
NUM_EPOCHS_A   = 5     # More epochs on tiny dataset (319 rows)
BATCH_SIZE_A   = 4
GRAD_ACCUM_A   = 4     # Effective batch = 16
LR_A           = 2e-4

print(f'Model:              {MODEL_A}')
print(f'Adapter output:     {ADAPTER_A}')
print(f'Effective batch:    {BATCH_SIZE_A * GRAD_ACCUM_A}')
print(f'Format:             Harmony (analysis + final channels)')

In [ ]:
# Cell 2-2: Load GPT-OSS-Safeguard with Unsloth 4-bit QLoRA
from unsloth import FastLanguageModel

model_a, tokenizer_a = FastLanguageModel.from_pretrained(
    model_name=MODEL_A,
    max_seq_length=MAX_SEQ_LEN_A,
    dtype=None,        # Auto: bfloat16 on A100/H100
    load_in_4bit=True,
)

print(f'Model loaded. dtype: {next(model_a.parameters()).dtype}')
print(f'Vocab size: {len(tokenizer_a)}')

In [ ]:
# Cell 2-3: Add LoRA adapters
model_a = FastLanguageModel.get_peft_model(
    model_a,
    r=LORA_R_A,
    lora_alpha=LORA_ALPHA_A,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj'],
)

trainable = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_a.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)')

In [ ]:
# Cell 2-4: Train
from trl import SFTTrainer, SFTConfig

training_args_a = SFTConfig(
    output_dir=str(ADAPTER_A / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS_A,
    per_device_train_batch_size=BATCH_SIZE_A,
    gradient_accumulation_steps=GRAD_ACCUM_A,
    per_device_eval_batch_size=BATCH_SIZE_A,
    warmup_ratio=0.1,
    learning_rate=LR_A,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    max_seq_length=MAX_SEQ_LEN_A,
    packing=False,
    dataset_text_field=None,
)

trainer_a = SFTTrainer(
    model=model_a,
    tokenizer=tokenizer_a,
    train_dataset=harmony_dataset['train'],
    eval_dataset=harmony_dataset['validation'],
    args=training_args_a,
)

print(f'Training {MODEL_A.split("/")[-1]} — {NUM_EPOCHS_A} epochs, {len(sg_train)} train rows')
trainer_a.train()

model_a.save_pretrained(str(ADAPTER_A))
tokenizer_a.save_pretrained(str(ADAPTER_A))
print(f'Adapter saved → {ADAPTER_A}')

In [ ]:
# Cell 2-5: Evaluate on SG test set
# GPT-OSS-Safeguard outputs in Harmony format — parse the <|start|>final<|message|>...<|end|> channel
import re
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm as tqdm_bar

FastLanguageModel.for_inference(model_a)


def parse_harmony_label(text: str) -> str:
    """Extract the label from the 'final' channel of a Harmony-formatted response."""
    match = re.search(r'<\|start\|>final<\|message\|>(.*?)<\|end\|>', text, re.DOTALL)
    if match:
        label = match.group(1).strip().lower()
        if label in LABEL_NAMES:
            return label
    # Fallback: scan raw output for known labels
    text_lower = text.lower()
    for label in LABEL_NAMES:
        if label in text_lower:
            return label
    return 'invalid'


def classify_gpt_oss(text, model, tokenizer, max_new_tokens=80):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Text: "{text}"'},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    return parse_harmony_label(generated)


print(f'Evaluating {MODEL_A.split("/")[-1]} on {len(sg_test)} test rows...')

y_true_a, y_pred_a = [], []
for row in tqdm_bar(sg_test):
    y_true_a.append(row['label'])
    y_pred_a.append(classify_gpt_oss(row['text'], model_a, tokenizer_a))

valid_a = [(t, p) for t, p in zip(y_true_a, y_pred_a) if p != 'invalid']
yt_a = [t for t, _ in valid_a]
yp_a = [p for _, p in valid_a]

print(f'\n{"="*60}')
print(f'  GPT-OSS-Safeguard-20B (fine-tuned) — SG Test Set')
print('='*60)
print(classification_report(yt_a, yp_a, labels=LABEL_NAMES, zero_division=0))
f1_a = f1_score(yt_a, yp_a, labels=LABEL_NAMES, average='macro', zero_division=0)
print(f'Macro F1: {f1_a:.4f}  |  Invalid outputs: {len(y_pred_a) - len(valid_a)}')

In [ ]:
# Cell 2-6: Save results + clear GPU
import csv, os
from datetime import datetime
from sklearn.metrics import classification_report as clf_report

report_a = clf_report(yt_a, yp_a, labels=LABEL_NAMES, output_dict=True, zero_division=0)
row_a = {
    'timestamp':     datetime.now().isoformat(),
    'model_name':    MODEL_A,
    'model_type':    'qlora (fine-tuned llm)',
    'task':          'sg_3class_lora',
    'format':        'harmony',
    'epochs':        NUM_EPOCHS_A,
    'lora_r':        LORA_R_A,
    'test_samples':  len(yt_a),
    'invalid_preds': len(y_pred_a) - len(valid_a),
    'accuracy':      round(report_a['accuracy'], 4),
    'macro_f1':      round(f1_a, 4),
    'hate_f1':       round(report_a.get('hate', {}).get('f1-score', 0), 4),
    'offensive_f1':  round(report_a.get('offensive', {}).get('f1-score', 0), 4),
    'normal_f1':     round(report_a.get('normal', {}).get('f1-score', 0), 4),
}

file_exists = os.path.exists(RESULTS_FILE)
with open(RESULTS_FILE, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row_a.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row_a)
print(f'Results saved → {RESULTS_FILE}')

import gc
del model_a, tokenizer_a, trainer_a
gc.collect()
torch.cuda.empty_cache()
print(f'GPU cleared. Free VRAM: {(gpu_mem_gb - torch.cuda.memory_reserved(0)/1e9):.1f} GB')

---
# Part 3: Gemma-SEA-LION-v4-27B-IT QLoRA

`aisingapore/Gemma-SEA-LION-v4-27B-IT` — 27B SEA-focused LLM built on Gemma 3 27B  
**License:** Gemma Terms of Use (permissive, no token required)  
**VRAM:** ~22 GB for 4-bit QLoRA  
**Unsloth:** Gemma 3 27B is fully supported — SEA-LION loads via same architecture  
**Fallback:** If direct load fails, uses `unsloth/gemma-3-27b-it-unsloth-bnb-4bit` as base  
**Estimated time:** ~50 min on A100 40GB, ~20 min on H100 80GB

In [ ]:
# Cell 3-1: Config
MODEL_B_PRIMARY  = 'aisingapore/Gemma-SEA-LION-v4-27B-IT'
MODEL_B_FALLBACK = 'unsloth/gemma-3-27b-it-unsloth-bnb-4bit'  # If SEA-LION direct load fails
ADAPTER_B        = OUTPUT_BASE / 'sea-lion-v4-27b-sg'
ADAPTER_B.mkdir(parents=True, exist_ok=True)

LORA_R_B      = 16   # Conservative for 27B dense model
LORA_ALPHA_B  = 16
MAX_SEQ_LEN_B = 512
NUM_EPOCHS_B  = 3
BATCH_SIZE_B  = 2    # 27B is larger — smaller batch
GRAD_ACCUM_B  = 8    # Effective batch = 16
LR_B          = 1e-4 # Lower LR for large dense model

print(f'Primary model:   {MODEL_B_PRIMARY}')
print(f'Fallback model:  {MODEL_B_FALLBACK}')
print(f'Adapter output:  {ADAPTER_B}')
print(f'Effective batch: {BATCH_SIZE_B * GRAD_ACCUM_B}')

In [ ]:
# Cell 3-2: Load Gemma-SEA-LION with Unsloth
# SEA-LION v4 27B is architecturally Gemma 3 27B — Unsloth supports it.
# We try the SEA-LION weights directly first (preferred: has SEA language pre-training).
# If that fails, we fall back to the Unsloth Gemma 3 27B 4-bit base.

from unsloth import FastLanguageModel

MODEL_B_USED = None

for model_id in [MODEL_B_PRIMARY, MODEL_B_FALLBACK]:
    try:
        print(f'Attempting to load: {model_id}')
        model_b, tokenizer_b = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=MAX_SEQ_LEN_B,
            dtype=None,
            load_in_4bit=True,
        )
        MODEL_B_USED = model_id
        print(f'Loaded: {model_id}')
        if model_id == MODEL_B_FALLBACK:
            print('NOTE: Using Gemma 3 27B base — SEA-LION weights not loaded.')
            print('      Results will reflect Gemma 3 base capability, not SEA-LION.')
        break
    except Exception as e:
        print(f'  Failed: {e}')
        print(f'  Trying fallback...')

if MODEL_B_USED is None:
    raise RuntimeError('Both model loads failed. Check your Colab runtime and Unsloth version.')

print(f'\ndtype: {next(model_b.parameters()).dtype}')

In [ ]:
# Cell 3-3: Add LoRA adapters
model_b = FastLanguageModel.get_peft_model(
    model_b,
    r=LORA_R_B,
    lora_alpha=LORA_ALPHA_B,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj'],
)

trainable = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_b.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)')

In [ ]:
# Cell 3-4: Train
from trl import SFTTrainer, SFTConfig

training_args_b = SFTConfig(
    output_dir=str(ADAPTER_B / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS_B,
    per_device_train_batch_size=BATCH_SIZE_B,
    gradient_accumulation_steps=GRAD_ACCUM_B,
    per_device_eval_batch_size=BATCH_SIZE_B,
    warmup_ratio=0.1,
    learning_rate=LR_B,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    max_seq_length=MAX_SEQ_LEN_B,
    packing=False,
    dataset_text_field=None,
)

trainer_b = SFTTrainer(
    model=model_b,
    tokenizer=tokenizer_b,
    train_dataset=chat_dataset['train'],      # Standard chat format
    eval_dataset=chat_dataset['validation'],
    args=training_args_b,
)

print(f'Training {MODEL_B_USED.split("/")[-1]} — {NUM_EPOCHS_B} epochs')
trainer_b.train()

model_b.save_pretrained(str(ADAPTER_B))
tokenizer_b.save_pretrained(str(ADAPTER_B))
print(f'Adapter saved → {ADAPTER_B}')

In [ ]:
# Cell 3-5: Evaluate
FastLanguageModel.for_inference(model_b)


def classify_chat_model(text, model, tokenizer, max_new_tokens=8):
    """Inference for standard chat-format models (SEA-LION, etc.)."""
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': f'Text: "{text}"'},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().lower()

    if raw.startswith('hate'):
        return 'hate'
    if raw.startswith('off'):
        return 'offensive'
    if raw.startswith('norm'):
        return 'normal'
    return 'invalid'


print(f'Evaluating {MODEL_B_USED.split("/")[-1]} on {len(sg_test)} test rows...')

y_true_b, y_pred_b = [], []
for row in tqdm_bar(sg_test):
    y_true_b.append(row['label'])
    y_pred_b.append(classify_chat_model(row['text'], model_b, tokenizer_b))

valid_b = [(t, p) for t, p in zip(y_true_b, y_pred_b) if p != 'invalid']
yt_b = [t for t, _ in valid_b]
yp_b = [p for _, p in valid_b]

print(f'\n{"="*60}')
print(f'  {MODEL_B_USED.split("/")[-1]} (fine-tuned) — SG Test Set')
print('='*60)
print(classification_report(yt_b, yp_b, labels=LABEL_NAMES, zero_division=0))
f1_b = f1_score(yt_b, yp_b, labels=LABEL_NAMES, average='macro', zero_division=0)
print(f'Macro F1: {f1_b:.4f}  |  Invalid: {len(y_pred_b) - len(valid_b)}')

In [ ]:
# Cell 3-6: Save results + clear GPU
report_b = clf_report(yt_b, yp_b, labels=LABEL_NAMES, output_dict=True, zero_division=0)
row_b = {
    'timestamp':     datetime.now().isoformat(),
    'model_name':    MODEL_B_USED,
    'model_type':    'qlora (fine-tuned llm)',
    'task':          'sg_3class_lora',
    'format':        'chat',
    'epochs':        NUM_EPOCHS_B,
    'lora_r':        LORA_R_B,
    'test_samples':  len(yt_b),
    'invalid_preds': len(y_pred_b) - len(valid_b),
    'accuracy':      round(report_b['accuracy'], 4),
    'macro_f1':      round(f1_b, 4),
    'hate_f1':       round(report_b.get('hate', {}).get('f1-score', 0), 4),
    'offensive_f1':  round(report_b.get('offensive', {}).get('f1-score', 0), 4),
    'normal_f1':     round(report_b.get('normal', {}).get('f1-score', 0), 4),
}

file_exists = os.path.exists(RESULTS_FILE)
with open(RESULTS_FILE, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row_b.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row_b)
print(f'Results saved → {RESULTS_FILE}')

del model_b, tokenizer_b, trainer_b
gc.collect()
torch.cuda.empty_cache()
print('GPU cleared.')

---
# Part 4: Leaderboard

Compare all fine-tuned models. Load baseline results from `llm_sg_baselines.ipynb` if available.

In [ ]:
import pandas as pd

# Collect from both possible result file locations
result_paths = [
    RESULTS_FILE,
    'results/model_scores_sg.csv',
]

dfs = []
for path in result_paths:
    if os.path.exists(path):
        dfs.append(pd.read_csv(path))

if not dfs:
    print('No results found yet.')
else:
    df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['model_name', 'task', 'format'], keep='last')
    df['short_model'] = df['model_name'].apply(lambda x: x.split('/')[-1])

    cols = ['short_model', 'model_type', 'task', 'format', 'macro_f1', 'hate_f1', 'offensive_f1', 'normal_f1', 'accuracy']
    existing_cols = [c for c in cols if c in df.columns]

    lb = df[existing_cols].sort_values('macro_f1', ascending=False).reset_index(drop=True)
    lb.index += 1

    print('SG Hate Speech — Full Leaderboard (Macro F1, 108 test rows)')
    print('='*100)
    print(lb.to_string())